In [2]:
import pandas as pd
from datetime import timedelta

# -------------------------------
# CONFIGURATION
# -------------------------------
AMOUNT_TOLERANCE = 50.0      # allowed difference
DATE_TOLERANCE_DAYS = 2      # allowed date difference

# -------------------------------
# LOAD DATA
# -------------------------------
def load_data(sap_file, sage_file):
    sap = pd.read_csv(sap_file, parse_dates=['Date'])
    sage = pd.read_csv(sage_file, parse_dates=['Date'])
    return sap, sage

# -------------------------------
# MATCHING LOGIC
# -------------------------------
def match_records(sap_df, sage_df):
    results = []

    sage_used = set()

    for idx, sap_row in sap_df.iterrows():
        matched = False

        for jdx, sage_row in sage_df.iterrows():

            if jdx in sage_used:
                continue

            # Match reference first (strong key)
            if sap_row['Reference'] == sage_row['Reference']:

                amount_diff = abs(sap_row['Amount'] - sage_row['Amount'])
                date_diff = abs((sap_row['Date'] - sage_row['Date']).days)

                if amount_diff == 0 and date_diff == 0:
                    status = "Perfect Match"

                elif amount_diff <= AMOUNT_TOLERANCE and date_diff <= DATE_TOLERANCE_DAYS:
                    status = "Partial Match (Tolerance)"

                else:
                    status = "Mismatch"

                results.append({
                    "SAP_Doc": sap_row['Document_ID'],
                    "SAGE_Doc": sage_row['Document_ID'],
                    "Reference": sap_row['Reference'],
                    "SAP_Amount": sap_row['Amount'],
                    "SAGE_Amount": sage_row['Amount'],
                    "Amount_Diff": amount_diff,
                    "Date_Diff_Days": date_diff,
                    "Status": status
                })

                sage_used.add(jdx)
                matched = True
                break

        if not matched:
            results.append({
                "SAP_Doc": sap_row['Document_ID'],
                "SAGE_Doc": None,
                "Reference": sap_row['Reference'],
                "SAP_Amount": sap_row['Amount'],
                "SAGE_Amount": None,
                "Amount_Diff": None,
                "Date_Diff_Days": None,
                "Status": "Missing in SAGE"
            })

    # Find SAGE records not matched
    unmatched_sage = sage_df.drop(index=list(sage_used))

    for _, row in unmatched_sage.iterrows():
        results.append({
            "SAP_Doc": None,
            "SAGE_Doc": row['Document_ID'],
            "Reference": row['Reference'],
            "SAP_Amount": None,
            "SAGE_Amount": row['Amount'],
            "Amount_Diff": None,
            "Date_Diff_Days": None,
            "Status": "Missing in SAP"
        })

    return pd.DataFrame(results)


# -------------------------------
# SUMMARY REPORT
# -------------------------------
def generate_summary(df):
    summary = df['Status'].value_counts().reset_index()
    summary.columns = ['Status', 'Count']
    return summary


# -------------------------------
# MAIN EXECUTION
# -------------------------------
def run_reconciliation(sap_file, sage_file, output_file="reconciliation_output.csv"):
    print("Loading data...")
    sap_df, sage_df = load_data(sap_file, sage_file)

    print("Running reconciliation...")
    result_df = match_records(sap_df, sage_df)

    print("Generating summary...")
    summary_df = generate_summary(result_df)

    print("\n=== RECONCILIATION SUMMARY ===")
    print(summary_df)

    print(f"\nSaving results to {output_file}...")
    result_df.to_csv(output_file, index=False)

    print("Done ✅")


# -------------------------------
# ENTRY POINT
# -------------------------------
if __name__ == "__main__":
    run_reconciliation(
        sap_file="/sap_sample.csv",
        sage_file="/sage_sample.csv"
    )

Loading data...
Running reconciliation...
Generating summary...

=== RECONCILIATION SUMMARY ===
                      Status  Count
0  Partial Match (Tolerance)     71
1              Perfect Match     22
2            Missing in SAGE      7

Saving results to reconciliation_output.csv...
Done ✅
